# Delta Lake — Hands-On Practice
### Day 3: Delta Tables, ACID Transactions, Time Travel, OPTIMIZE, Z-ORDER, VACUUM

**Prerequisites:** This notebook assumes Databricks Free Edition (serverless-only compute).
No cluster configuration is required — just attach this notebook to a serverless SQL
warehouse or serverless compute and run the cells top to bottom.

**How to use this notebook:**
- Run each cell in order — later sections depend on tables created earlier.
- Cells marked **"Try it"** are meant for you to modify and re-run, not just read.
- Cells marked **"Watch this fail"** are intentional errors — read the error message, don't skip past it.

## 0. Setup
Create a dedicated catalog/schema so your practice tables don't clutter shared workspace data.

In [0]:
# Use your workspace's default catalog if 'main' isn't available, e.g. spark.sql("USE CATALOG hive_metastore")
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("USE CATALOG main")
spark.sql("CREATE SCHEMA IF NOT EXISTS delta_practice")
spark.sql("USE SCHEMA delta_practice")

print("Active catalog/schema:")
spark.sql("SELECT current_catalog(), current_schema()").show()

## 1. Creating Delta Tables
Three common ways to create a Delta table. All three produce the same underlying format —
a directory of Parquet data files plus a `_delta_log` transaction log.

### 1a. SQL — CREATE TABLE ... USING DELTA

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE orders (
    order_id     INT,
    customer_id  INT,
    order_status STRING,
    order_amount DOUBLE,
    order_date   DATE
) USING DELTA
""")

spark.sql("""
INSERT INTO orders VALUES
    (1, 101, 'PLACED',   250.00, '2026-07-01'),
    (2, 102, 'PLACED',   99.50,  '2026-07-01'),
    (3, 103, 'SHIPPED',  480.00, '2026-07-02'),
    (4, 101, 'DELIVERED',120.00, '2026-07-03'),
    (5, 104, 'CANCELLED',60.00,  '2026-07-03')
""")

spark.sql("SELECT * FROM orders ORDER BY order_id").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE orders1 (
    order_id     INT,
    customer_id  INT,
    order_status STRING,
    order_amount DOUBLE,
    order_date   DATE
) 
""")

spark.sql("""
INSERT INTO orders1 VALUES
    (1, 101, 'PLACED',   250.00, '2026-07-01'),
    (2, 102, 'PLACED',   99.50,  '2026-07-01'),
    (3, 103, 'SHIPPED',  480.00, '2026-07-02'),
    (4, 101, 'DELIVERED',120.00, '2026-07-03'),
    (5, 104, 'CANCELLED',60.00,  '2026-07-03')
""")

spark.sql("SELECT * FROM orders1 ORDER BY order_id").show()

In [0]:
DESCRIBE orders;

### 1b. Python DataFrame API — write.format("delta")

In [0]:
from pyspark.sql import Row

customers_data = [
    Row(customer_id=101, name="Asha Rao",     city="Chennai",   signup_date="2025-11-01"),
    Row(customer_id=102, name="Vikram Shah",  city="Mumbai",    signup_date="2025-12-15"),
    Row(customer_id=103, name="Priya Nair",   city="Bengaluru", signup_date="2026-01-20"),
    Row(customer_id=104, name="Karan Mehta",  city="Delhi",     signup_date="2026-02-10"),
]

customers_df = spark.createDataFrame(customers_data)

(customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customers"))

spark.sql("SELECT * FROM customers ORDER BY customer_id").show()

### 1c. Converting an existing source (CSV/Parquet) into Delta
In real pipelines, source data often lands as CSV or Parquet first. Converting to Delta
gets you ACID transactions, time travel, and performance features on top of it.

In [0]:
# Simulate a "raw" CSV landing zone using DBFS
raw_path = "/Volumes/main/uc_demo/raw_files/Test_20260515_094925.csv"

products_data = [
    (1, "Wireless Mouse", "Electronics", 799.0),
    (2, "Office Chair",   "Furniture",   4500.0),
    (3, "Notebook Pack",  "Stationery",  150.0),
]
products_df = spark.createDataFrame(products_data, ["product_id", "product_name", "category", "price"])
products_df.write.mode("overwrite").option("header", True).csv(raw_path)

# Read CSV, write out as Delta -> this IS the "conversion"
delta_products_path = "/tmp/delta_practice/products_delta"

raw_df = spark.read.option("header", True).option("inferSchema", True).csv(raw_path)
raw_df.write.format("delta").mode("overwrite").save(delta_products_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS products USING DELTA LOCATION '{delta_products_path}'")
spark.sql("SELECT * FROM products").show()

**Try it:** Create a fourth table of your own (any subject — e.g. `warehouses`, `employees`)
using whichever of the three methods above you find easiest to remember.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE student (
    order_id     INT,
    customer_id  INT,
    order_status STRING,
    order_amount DOUBLE,
    order_date   DATE
) USING DELTA
""")

spark.sql("""
INSERT INTO orders VALUES
    (1, 101, 'PLACED',   250.00, '2026-07-01'),
    (2, 102, 'PLACED',   99.50,  '2026-07-01'),
    (3, 103, 'SHIPPED',  480.00, '2026-07-02'),
    (4, 101, 'DELIVERED',120.00, '2026-07-03'),
    (5, 104, 'CANCELLED',60.00,  '2026-07-03')
""")

spark.sql("SELECT * FROM orders ORDER BY order_id").show()

## 2. ACID Transactions in Action
Delta Lake gives every write — INSERT, UPDATE, DELETE, MERGE — full ACID guarantees.
If a write fails halfway through, the table is left exactly as it was before (Atomicity),
and readers never see a half-written state (Isolation).

### 2a. UPDATE

In [0]:
spark.sql("UPDATE orders SET order_status = 'DELIVERED' WHERE order_id = 3")
spark.sql("SELECT * FROM orders ORDER BY order_id").show()

In [0]:
spark.sql("UPDATE orders1 SET order_status = 'DELIVERED' WHERE order_id = 3")
spark.sql("SELECT * FROM orders1 ORDER BY order_id").show()

### 2b. DELETE

In [0]:
spark.sql("DELETE FROM orders WHERE order_status = 'CANCELLED'")
spark.sql("SELECT * FROM orders ORDER BY order_id").show()

In [0]:
%sql
DESCRIBE HISTORY orders

### 2c. MERGE INTO (upsert) — the workhorse of real pipelines
Simulates a daily batch of order updates: some are new orders, some update existing ones.

In [0]:
incoming_data = [
    (2, 102, 'DELIVERED', 99.50,  '2026-07-01'),  # existing order_id -> should UPDATE
    (6, 105, 'PLACED',    310.00, '2026-07-04'),  # new order_id -> should INSERT
]
incoming_df = spark.createDataFrame(
    incoming_data, ["order_id", "customer_id", "order_status", "order_amount", "order_date"]
)
incoming_df.createOrReplaceTempView("incoming_orders")

spark.sql("""
MERGE INTO orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
    UPDATE SET target.order_status = source.order_status
WHEN NOT MATCHED THEN
    INSERT *
""")

spark.sql("SELECT * FROM orders ORDER BY order_id").show()

**Try it:** Add a third row to `incoming_orders` that updates `order_amount` instead of
`order_status`, and extend the `MERGE` statement's `UPDATE SET` clause to handle it.

In [0]:
# Your code here

## 3. Schema Enforcement — Watch This Fail
Delta Lake rejects writes that don't match the table's schema by default. This protects
you from silently corrupting a table with unexpected columns or types.

In [0]:
%sql
select * from orders;

In [0]:
bad_data = [(7, 106, 200.00)]  # extra column!
bad_df = spark.createDataFrame(
    bad_data, ["order_id", "customer_id", "order_status", "order_amount" ]
)

try:
    bad_df.write.mode("append").saveAsTable("orders")
except Exception as e:
    print("Write REJECTED, as expected. Error summary:")
    print(str(e)[:500])

In [0]:
%sql
select * from orders;

In [0]:
bad_data = [(7, 106, 'PLACED', 200.00, '2026-07-05', 'EXPRESS_SHIPPING')]  # extra column!
bad_df = spark.createDataFrame(
    bad_data, ["order_id", "customer_id", "order_status", "order_amount", "order_date", "shipping_type"]
)

try:
    bad_df.write.mode("append").option("mergeSchema", "true").saveAsTable("orders1")
except Exception as e:
    print("Write REJECTED, as expected. Error summary:")
    print(str(e)[:500])

## 4. Schema Evolution — Doing It on Purpose
If you actually *want* to add a new column, you have two options: alter the table first,
or let Delta merge the new schema automatically with `mergeSchema`.

### 4a. ALTER TABLE — explicit column addition

In [0]:
spark.sql("ALTER TABLE orders ADD COLUMN shipping_type STRING")
spark.sql("SELECT * FROM orders ORDER BY order_id").show()

### 4b. mergeSchema — let the write itself evolve the schema

In [0]:
evolved_data = [(8, 107, 'PLACED', 175.00, '2026-07-05', 'STANDARD', 'Chennai')]  # new column: delivery_city
evolved_df = spark.createDataFrame(
    evolved_data,
    ["order_id", "customer_id", "order_status", "order_amount", "order_date", "shipping_type", "delivery_city"]
)

(evolved_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("orders"))

spark.sql("SELECT * FROM orders ORDER BY order_id").show()

**Try it:** Now that `orders` has a `delivery_city` column, what happens to the *older*
rows (order_id 1–7) that were written before this column existed? Query the table and
check — then explain in one sentence why that's the correct/expected behavior.

In [0]:
# Your code here

## 5. Time Travel
Every write creates a new table version in the Delta transaction log. You can query,
compare, or restore any past version.

### 5a. View the full history

In [0]:
spark.sql("DESCRIBE HISTORY orders").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

### 5b. Query a past version — VERSION AS OF

In [0]:
# Version 0 = the table right after it was first created (before UPDATE/DELETE/MERGE/ALTER)
spark.sql("SELECT * FROM orders VERSION AS OF 0 ORDER BY order_id").show()

### 5c. Query as of a timestamp
Replace the timestamp below with one copied from the `DESCRIBE HISTORY` output above.

In [0]:
# Example — update this timestamp string to match your own history output
# spark.sql("SELECT * FROM orders TIMESTAMP AS OF '2026-07-22 10:00:00' ORDER BY order_id").show()

print("Update the timestamp above using a value from Section 5a's output, then uncomment and run.")

### 5d. RESTORE TABLE — roll the table back to a past version
Unlike a `SELECT ... VERSION AS OF` (a read-only query), `RESTORE` actually rewrites the
table's current state to match a past version. Use with care — this is itself a new
transaction, so you can always time-travel to undo a restore too.

In [0]:
# Restore to version 0 (before any of our UPDATE/DELETE/MERGE/schema changes)
spark.sql("RESTORE TABLE orders TO VERSION AS OF 0")
spark.sql("SELECT * FROM orders ORDER BY order_id").show()

print("Table restored. Check DESCRIBE HISTORY -> the RESTORE itself is logged as a new version.")
spark.sql("DESCRIBE HISTORY orders").select("version", "operation").show()

**Try it:** Re-run the MERGE from Section 2c so the table is back to its evolved state
(you'll need it for the OPTIMIZE section below). Then check `DESCRIBE HISTORY` again —
how many versions does the table have now?

In [0]:
# Your code here

## 6. OPTIMIZE & Z-ORDER
Frequent small writes (like the ones we've been doing) create many small files, which
slows down reads. `OPTIMIZE` compacts them into fewer, larger files.

### 6a. Create a deliberately fragmented table
We'll append data one tiny batch at a time to simulate a streaming/frequent-write pattern.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE sensor_readings (
    sensor_id   INT,
    reading_ts  TIMESTAMP,
    metric_name STRING,
    value       DOUBLE
) USING DELTA
""")

import random
from datetime import datetime, timedelta

base_time = datetime(2026, 7, 20, 0, 0, 0)
metrics = ["temperature", "vibration", "pressure"]

# 25 tiny appends -> 25 small files, on purpose
for i in range(25):
    batch = [
        (random.randint(1, 5), base_time + timedelta(minutes=i), random.choice(metrics), round(random.uniform(10, 100), 2))
        for _ in range(5)
    ]
    batch_df = spark.createDataFrame(batch, ["sensor_id", "reading_ts", "metric_name", "value"])
    batch_df.write.format("delta").mode("append").saveAsTable("sensor_readings")

print("25 small append batches written.")

### 6b. Check file count before OPTIMIZE

In [0]:
detail_before = spark.sql("DESCRIBE DETAIL sensor_readings").select("numFiles", "sizeInBytes").collect()[0]
print(f"Before OPTIMIZE -> files: {detail_before['numFiles']}, size: {detail_before['sizeInBytes']} bytes")

### 6c. Run OPTIMIZE and compare

In [0]:
spark.sql("OPTIMIZE sensor_readings")

detail_after = spark.sql("DESCRIBE DETAIL sensor_readings").select("numFiles", "sizeInBytes").collect()[0]
print(f"After OPTIMIZE  -> files: {detail_after['numFiles']}, size: {detail_after['sizeInBytes']} bytes")
print("Note: OPTIMIZE compacts files but keeps old files around until VACUUM removes them -> see Section 7.")

### 6d. OPTIMIZE with Z-ORDER
Z-ORDER co-locates related data in the same files based on the column(s) you specify —
useful when queries frequently filter on that column (here, `sensor_id`).

In [0]:
spark.sql("OPTIMIZE sensor_readings ZORDER BY (sensor_id)")

print("Z-ORDER complete. Queries filtering on sensor_id (e.g. WHERE sensor_id = 3) can now skip")
print("more files entirely, since matching rows are physically clustered together.")

spark.sql("SELECT * FROM sensor_readings WHERE sensor_id = 3 LIMIT 5").show()

**Try it:** Which column would you Z-ORDER the `orders` table on, and why? (Hint: think
about which column your team would most often filter or join on in a real query.)
Run it once you've picked one.

In [0]:
# Your code here

## 7. VACUUM
OPTIMIZE and other operations leave old data files behind (that's what makes time travel
possible). VACUUM permanently deletes files older than the retention threshold.

**Warning:** Once VACUUM removes a file, you lose the ability to time-travel to any
version that depended on it. The default retention is 7 days — the command below uses a
shorter window purely for this practice exercise, which requires disabling a safety check.

### 7a. Dry run first — always do this before a real VACUUM

In [0]:
spark.sql("VACUUM sensor_readings DRY RUN")
# This lists the files that WOULD be deleted, without actually deleting anything.

### 7b. Real VACUUM (practice-only short retention)
In production, never disable the retention duration check like this — 7 days is the
safe default. We're only doing it here so you can see the effect immediately.

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM sensor_readings RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("VACUUM complete. Old small files from before OPTIMIZE are now permanently removed.")

**Try it:** After this VACUUM, try time-traveling `sensor_readings` back to an early
version (e.g. `VERSION AS OF 1`). What happens, and why does that make sense given what
VACUUM just did?

In [0]:
# Your code here

## 8. Bonus — Stretch Practice
Optional, for anyone who finishes early.

### 8a. Table constraints

In [0]:
spark.sql("ALTER TABLE orders ADD CONSTRAINT positive_amount CHECK (order_amount >= 0)")

try:
    spark.sql("INSERT INTO orders VALUES (99, 108, 'PLACED', -50.00, '2026-07-06', 'STANDARD', NULL)")
except Exception as e:
    print("Rejected by constraint, as expected:")
    print(str(e)[:300])

### 8b. Generated columns
A column whose value Delta computes automatically from other columns.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE orders_with_year (
    order_id     INT,
    order_date   DATE,
    order_year   INT GENERATED ALWAYS AS (YEAR(order_date))
) USING DELTA
""")

spark.sql("INSERT INTO orders_with_year (order_id, order_date) VALUES (1, '2026-07-22')")
spark.sql("SELECT * FROM orders_with_year").show()

### 8c. CLONE — shallow vs deep
Shallow clone copies only metadata (fast, points back at original data files) —
useful for quickly spinning up a test/dev copy of a table.

In [0]:
spark.sql("CREATE OR REPLACE TABLE orders_test_clone SHALLOW CLONE orders")
spark.sql("SELECT COUNT(*) AS row_count FROM orders_test_clone").show()

print("Shallow clone created. Try modifying orders_test_clone and confirm the original 'orders'")
print("table is unaffected -- that's the isolation a clone gives you for safe experimentation.")

## 9. Wrap-Up Challenge
Combine everything from this notebook, without looking back at the cells above:

1. Create a new Delta table called `inventory` with at least 3 columns and 5 rows.
2. Write a `MERGE INTO` that updates 2 existing rows and inserts 2 new ones in one statement.
3. Try an append with a mismatched schema and confirm it's rejected.
4. Add a new column using `mergeSchema`.
5. Check `DESCRIBE HISTORY` and query an earlier version with `VERSION AS OF`.
6. Run `OPTIMIZE ... ZORDER BY` on a column of your choice and explain your choice in a comment.
7. Run a `VACUUM DRY RUN` (do not disable the retention check this time — just observe the dry run output).

No solution cells below on purpose — this one's yours.

In [0]:
# Your solution here